In [1]:
from pathlib import Path
import os
import sys
import json
import zipfile
import shutil
import logging
import urllib.request
import random
import requests
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name != "fruitnet-chili-yield":
    if PROJECT_ROOT.parent.name == "fruitnet-chili-yield":
        PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

assert PROJECT_ROOT.name == "fruitnet-chili-yield", (
    f"Please run this script from the fruitnet-chili-yield root directory. Current: {PROJECT_ROOT}"
)

os.chdir(PROJECT_ROOT)

# ============================================================
# DATA STORAGE LOCATION FOR WSL
# Windows drive F: is mounted in WSL as /mnt/f
#
# Default dataset folder:
#   Windows: F:\\fruitnet-chili-yield-data
#   WSL:     /mnt/f/fruitnet-chili-yield-data
#
# You can override it before running:
#   export FRUITNET_DATA_ROOT=/mnt/f/your-folder-name
# ============================================================
DATA_ROOT = Path(
    os.environ.get("FRUITNET_DATA_ROOT", "/mnt/f/fruitnet-chili-yield-data")
).expanduser().resolve()

DATA_RAW = DATA_ROOT / "raw"
DATA_PROCESSED = DATA_ROOT / "processed"

for d in [
    Path("logs"),
    DATA_RAW / "coco2017",
    DATA_RAW / "minneapple",
    DATA_RAW / "gwhd",
    DATA_PROCESSED / "coco_yolo",
    DATA_PROCESSED / "minneapple_yolo",
    DATA_PROCESSED / "gwhd_yolo",
    Path("configs/data"),
    Path("results/dataset_preparation"),
]:
    d.mkdir(parents=True, exist_ok=True)

LOG_PATH = Path("logs") / f"01_download_prepare_datasets_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()

_log_fp = open(LOG_PATH, "a", encoding="utf-8")
sys.stdout = Tee(sys.__stdout__, _log_fp)
sys.stderr = Tee(sys.__stderr__, _log_fp)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH, encoding="utf-8"),
        logging.StreamHandler(sys.__stdout__),
    ],
)

logging.info("PROJECT_ROOT=%s", PROJECT_ROOT)
logging.info("DATA_ROOT=%s", DATA_ROOT)
logging.info("Log file=%s", LOG_PATH)

DOWNLOAD_COCO = True
CONVERT_COCO_TO_YOLO = True
DOWNLOAD_MINNEAPPLE = False
MINNEAPPLE_URL = os.environ.get("MINNEAPPLE_URL", "").strip()
DOWNLOAD_GWHD = False
GWHD_ZENODO_RECORD = os.environ.get("GWHD_ZENODO_RECORD", "5092309")

COCO_RAW = DATA_RAW / "coco2017"
COCO_PROC = DATA_PROCESSED / "coco_yolo"

MINNE_RAW = DATA_RAW / "minneapple"
MINNE_PROC = DATA_PROCESSED / "minneapple_yolo"

GWHD_RAW = DATA_RAW / "gwhd"
GWHD_PROC = DATA_PROCESSED / "gwhd_yolo"

print("DATA_ROOT:", DATA_ROOT)
print("DOWNLOAD_COCO:", DOWNLOAD_COCO)
print("DOWNLOAD_MINNEAPPLE:", DOWNLOAD_MINNEAPPLE)
print("MINNEAPPLE_URL exists:", bool(MINNEAPPLE_URL))
print("DOWNLOAD_GWHD:", DOWNLOAD_GWHD)
print("GWHD_ZENODO_RECORD:", GWHD_ZENODO_RECORD)

def download_file(url: str, out_path: Path, chunk_size: int = 1024 * 1024) -> Path:
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if out_path.exists() and out_path.stat().st_size > 0:
        logging.info("Skipped existing file: %s", out_path)
        return out_path

    logging.info("Downloading %s -> %s", url, out_path)

    with urllib.request.urlopen(url) as response:
        total = int(response.headers.get("Content-Length", 0))

        with open(out_path, "wb") as f, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=out_path.name,
        ) as pbar:
            while True:
                chunk = response.read(chunk_size)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))

    return out_path

def extract_zip(zip_path: Path, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    marker = out_dir / f".extracted_{zip_path.stem}"

    if marker.exists():
        logging.info("Skipped extraction, marker exists: %s", marker)
        return

    logging.info("Extracting %s -> %s", zip_path, out_dir)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

    marker.write_text(str(datetime.now()), encoding="utf-8")

def write_yaml(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.strip() + "\n", encoding="utf-8")
    logging.info("Wrote YAML file: %s", path)

def xywh_to_yolo(x, y, w, h, img_w, img_h):
    xc = (x + w / 2) / img_w
    yc = (y + h / 2) / img_h
    ww = w / img_w
    hh = h / img_h
    return xc, yc, ww, hh

def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    w = max(0.0, x2 - x1)
    h = max(0.0, y2 - y1)
    return xywh_to_yolo(x1, y1, w, h, img_w, img_h)

def safe_link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src.resolve(), dst)
    except Exception:
        shutil.copy2(src, dst)

COCO_URLS = {
    "train_images": "http://images.cocodataset.org/zips/train2017.zip",
    "val_images": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
}

if DOWNLOAD_COCO:
    for name, url in COCO_URLS.items():
        download_file(url, COCO_RAW / f"{name}.zip")

    extract_zip(COCO_RAW / "train_images.zip", COCO_RAW)
    extract_zip(COCO_RAW / "val_images.zip", COCO_RAW)
    extract_zip(COCO_RAW / "annotations.zip", COCO_RAW)

def convert_coco_split(split: str):
    ann_path = COCO_RAW / "annotations" / f"instances_{split}2017.json"
    img_dir = COCO_RAW / f"{split}2017"

    if not ann_path.exists():
        logging.warning("COCO annotations not found: %s", ann_path)
        return {"split": split, "status": "missing_ann"}

    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    img_by_id = {im["id"]: im for im in coco["images"]}
    cats = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_contiguous = {c["id"]: i for i, c in enumerate(cats)}
    names = {i: c["name"] for i, c in enumerate(cats)}

    out_img_dir = COCO_PROC / "images" / f"{split}2017"
    out_lab_dir = COCO_PROC / "labels" / f"{split}2017"

    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lab_dir.mkdir(parents=True, exist_ok=True)

    labels = {im_id: [] for im_id in img_by_id.keys()}

    for ann in tqdm(coco["annotations"], desc=f"Processing COCO {split} annotations"):
        if ann.get("iscrowd", 0) == 1:
            continue

        im = img_by_id[ann["image_id"]]
        cls = cat_id_to_contiguous[ann["category_id"]]
        x, y, w, h = ann["bbox"]

        if w <= 1 or h <= 1:
            continue

        xc, yc, ww, hh = xywh_to_yolo(x, y, w, h, im["width"], im["height"])
        vals = [xc, yc, ww, hh]
        vals = [min(max(v, 0.0), 1.0) for v in vals]

        labels[ann["image_id"]].append(f"{cls} " + " ".join(f"{v:.6f}" for v in vals))

    n_images = 0
    n_labels = 0

    for im_id, im in tqdm(img_by_id.items(), desc=f"Linking COCO {split} images and labels"):
        src = img_dir / im["file_name"]
        dst = out_img_dir / im["file_name"]

        if src.exists():
            safe_link_or_copy(src, dst)

        label_path = out_lab_dir / (Path(im["file_name"]).stem + ".txt")
        lines = labels.get(im_id, [])
        label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

        n_images += 1
        n_labels += len(lines)

    names_path = COCO_PROC / "coco_names.json"
    names_path.write_text(json.dumps(names, indent=2), encoding="utf-8")

    return {
        "split": split,
        "status": "ok",
        "images": n_images,
        "labels": n_labels,
    }

coco_summary = []

if CONVERT_COCO_TO_YOLO:
    coco_summary.append(convert_coco_split("train"))
    coco_summary.append(convert_coco_split("val"))

pd.DataFrame(coco_summary).to_csv(
    "results/dataset_preparation/coco_conversion_summary.csv", index=False
)

if DOWNLOAD_MINNEAPPLE:
    if not MINNEAPPLE_URL:
        logging.warning("MINNEAPPLE_URL is empty. Please download it manually and place the files in %s", MINNE_RAW)
    else:
        archive_name = MINNEAPPLE_URL.split("/")[-1] or "minneapple.zip"
        archive = download_file(MINNEAPPLE_URL, MINNE_RAW / archive_name)
        if archive.suffix.lower() == ".zip":
            extract_zip(archive, MINNE_RAW)

def find_first_json(root: Path):
    candidates = list(root.rglob("*.json"))
    for p in candidates:
        if "instance" in p.name.lower() or "annotation" in p.name.lower():
            return p
    return candidates[0] if candidates else None

def convert_coco_like_single_class_dataset(raw_root: Path, proc_root: Path, dataset_name: str, class_name: str):
    proc_root.mkdir(parents=True, exist_ok=True)
    json_path = find_first_json(raw_root)

    if json_path is None:
        logging.warning("%s: No JSON found in %s. If the dataset is already in YOLO format, place it in %s", dataset_name, raw_root, proc_root)
        return {"dataset": dataset_name, "status": "missing_json"}

    logging.info("%s: Converting JSON %s", dataset_name, json_path)
    data = json.loads(json_path.read_text(encoding="utf-8"))

    if "images" not in data or "annotations" not in data:
        logging.warning("%s: JSON format is not COCO-like: %s", dataset_name, json_path)
        return {"dataset": dataset_name, "status": "not_coco_like", "json": str(json_path)}

    image_by_id = {im["id"]: im for im in data["images"]}
    anns_by_img = {im_id: [] for im_id in image_by_id.keys()}

    for ann in data["annotations"]:
        if "bbox" not in ann:
            continue
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    image_files = []
    for im in data["images"]:
        fname = im.get("file_name", "")
        matches = list(raw_root.rglob(Path(fname).name))
        if matches:
            image_files.append((im, matches[0]))

    random.seed(42)
    random.shuffle(image_files)

    n_train = int(len(image_files) * 0.8)
    splits = {
        "train": image_files[:n_train],
        "val": image_files[n_train:],
    }

    summary = []
    for split, items in splits.items():
        img_out = proc_root / "images" / split
        lab_out = proc_root / "labels" / split

        img_out.mkdir(parents=True, exist_ok=True)
        lab_out.mkdir(parents=True, exist_ok=True)

        n_boxes = 0
        for im, img_src in tqdm(items, desc=f"Processing {dataset_name} {split}"):
            img_dst = img_out / img_src.name
            safe_link_or_copy(img_src, img_dst)

            img_w = im.get("width")
            img_h = im.get("height")

            if img_w is None or img_h is None:
                with Image.open(img_src) as img:
                    img_w, img_h = img.size

            lines = []
            for ann in anns_by_img.get(im["id"], []):
                x, y, w, h = ann["bbox"]
                if w <= 1 or h <= 1:
                    continue

                xc, yc, ww, hh = xywh_to_yolo(x, y, w, h, img_w, img_h)
                vals = [xc, yc, ww, hh]
                vals = [min(max(v, 0.0), 1.0) for v in vals]
                lines.append("0 " + " ".join(f"{v:.6f}" for v in vals))

            (lab_out / (img_src.stem + ".txt")).write_text(
                "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8"
            )
            n_boxes += len(lines)

        summary.append({
            "dataset": dataset_name,
            "split": split,
            "images": len(items),
            "boxes": n_boxes,
        })

    write_yaml(
        Path("configs/data") / f"{dataset_name.lower()}.yaml",
        f"""
path: {proc_root.resolve()}
train: images/train
val: images/val
test: images/val
names:
  0: {class_name}
""",
    )

    return {
        "dataset": dataset_name,
        "status": "ok",
        "json": str(json_path),
        "images": len(image_files),
        "summary": summary,
    }

minne_summary = convert_coco_like_single_class_dataset(
    raw_root=MINNE_RAW,
    proc_root=MINNE_PROC,
    dataset_name="minneapple",
    class_name="apple",
)

Path("results/dataset_preparation/minneapple_conversion_summary.json").write_text(
    json.dumps(minne_summary, indent=2), encoding="utf-8"
)

def zenodo_download_record(record_id: str, out_dir: Path):
    api_url = f"https://zenodo.org/api/records/{record_id}"
    logging.info("Querying Zenodo API: %s", api_url)

    r = requests.get(api_url, timeout=60)
    r.raise_for_status()
    record = r.json()
    files = record.get("files", [])

    if not files:
        logging.warning("No files found in Zenodo record %s", record_id)
        return []

    downloaded = []
    for f in files:
        key = f.get("key") or f.get("filename") or "zenodo_file"
        links = f.get("links", {})
        url = links.get("self") or links.get("download")
        if not url:
            continue
        out_path = out_dir / key
        downloaded.append(download_file(url, out_path))

    return downloaded

if DOWNLOAD_GWHD:
    files = zenodo_download_record(GWHD_ZENODO_RECORD, GWHD_RAW)
    for p in files:
        if p.suffix.lower() == ".zip":
            extract_zip(p, GWHD_RAW)

def parse_gwhd_boxes_string(s):
    if pd.isna(s):
        return []

    s = str(s).strip()
    if s == "" or s.lower() == "no_box":
        return []

    boxes = []
    for part in s.split(";"):
        part = part.strip()
        if not part:
            continue

        nums = [
            float(x)
            for x in part.replace("[", "").replace("]", "").replace(",", " ").split()
        ]
        if len(nums) >= 4:
            boxes.append(nums[:4])

    return boxes

def convert_gwhd_csv_to_yolo(raw_root: Path, proc_root: Path):
    csv_candidates = list(raw_root.rglob("competition_train.csv")) + list(raw_root.rglob("train.csv"))
    val_candidates = list(raw_root.rglob("competition_val.csv")) + list(raw_root.rglob("val.csv"))

    if not csv_candidates:
        logging.warning("GWHD training CSV not found in %s", raw_root)
        return {"dataset": "gwhd", "status": "missing_csv"}

    train_csv = csv_candidates[0]
    val_csv = val_candidates[0] if val_candidates else train_csv

    split_csvs = {
        "train": train_csv,
        "val": val_csv,
    }

    image_candidates = []
    for ext in ["*.png", "*.jpg", "*.jpeg"]:
        image_candidates.extend(raw_root.rglob(ext))

    image_by_stem = {p.stem: p for p in image_candidates}
    summary = []

    for split, csv_path in split_csvs.items():
        df = pd.read_csv(csv_path)
        img_out = proc_root / "images" / split
        lab_out = proc_root / "labels" / split

        img_out.mkdir(parents=True, exist_ok=True)
        lab_out.mkdir(parents=True, exist_ok=True)

        n_images = 0
        n_boxes = 0

        image_col = "image_name" if "image_name" in df.columns else df.columns[0]
        boxes_col = "BoxesString" if "BoxesString" in df.columns else None

        if boxes_col is None:
            for c in df.columns:
                if "box" in c.lower():
                    boxes_col = c
                    break

        if boxes_col is None:
            raise ValueError(f"Cannot find boxes column in {csv_path}. Available columns: {df.columns.tolist()}")

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing GWHD {split}"):
            stem = str(row[image_col]).replace(".png", "").replace(".jpg", "").replace(".jpeg", "")
            src = image_by_stem.get(stem)

            if src is None:
                continue

            safe_link_or_copy(src, img_out / src.name)
            with Image.open(src) as img:
                img_w, img_h = img.size

            lines = []
            for x, y, w, h in parse_gwhd_boxes_string(row[boxes_col]):
                xc, yc, ww, hh = xywh_to_yolo(x, y, w, h, img_w, img_h)
                vals = [xc, yc, ww, hh]
                vals = [min(max(v, 0.0), 1.0) for v in vals]
                lines.append("0 " + " ".join(f"{v:.6f}" for v in vals))

            (lab_out / (src.stem + ".txt")).write_text(
                "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8"
            )

            n_images += 1
            n_boxes += len(lines)

        summary.append({
            "split": split,
            "csv": str(csv_path),
            "images": n_images,
            "boxes": n_boxes,
        })

    write_yaml(
        Path("configs/data/gwhd.yaml"),
        f"""
path: {proc_root.resolve()}
train: images/train
val: images/val
test: images/val
names:
  0: wheat_head
""",
    )

    return {"dataset": "gwhd", "status": "ok", "summary": summary}

gwhd_summary = convert_gwhd_csv_to_yolo(GWHD_RAW, GWHD_PROC)

Path("results/dataset_preparation/gwhd_conversion_summary.json").write_text(
    json.dumps(gwhd_summary, indent=2), encoding="utf-8"
)

coco_names_path = COCO_PROC / "coco_names.json"

if coco_names_path.exists():
    names = json.loads(coco_names_path.read_text(encoding="utf-8"))
else:
    names = {i: f"class_{i}" for i in range(80)}

names_yaml = "\n".join([f"  {int(k)}: {v}" for k, v in names.items()])

write_yaml(
    Path("configs/data/coco.yaml"),
    f"""
path: {COCO_PROC.resolve()}
train: images/train2017
val: images/val2017
test: images/val2017
names:
{names_yaml}
""",
)

dataset_status = {
    "coco_yaml": str(Path("configs/data/coco.yaml").resolve()),
    "minneapple_yaml": str(Path("configs/data/minneapple.yaml").resolve()),
    "gwhd_yaml": str(Path("configs/data/gwhd.yaml").resolve()),
    "coco_train_images": len(list((COCO_PROC / "images/train2017").glob("*"))) if (COCO_PROC / "images/train2017").exists() else 0,
    "coco_val_images": len(list((COCO_PROC / "images/val2017").glob("*"))) if (COCO_PROC / "images/val2017").exists() else 0,
    "minneapple_train_images": len(list((MINNE_PROC / "images/train").glob("*"))) if (MINNE_PROC / "images/train").exists() else 0,
    "minneapple_val_images": len(list((MINNE_PROC / "images/val").glob("*"))) if (MINNE_PROC / "images/val").exists() else 0,
    "gwhd_train_images": len(list((GWHD_PROC / "images/train").glob("*"))) if (GWHD_PROC / "images/train").exists() else 0,
    "gwhd_val_images": len(list((GWHD_PROC / "images/val").glob("*"))) if (GWHD_PROC / "images/val").exists() else 0,
    "log_file": str(LOG_PATH),
}

Path("results/dataset_preparation/dataset_status.json").write_text(
    json.dumps(dataset_status, indent=2), encoding="utf-8"
)

print(json.dumps(dataset_status, indent=2))
print("Log file:", LOG_PATH)

/home/diy-hus/miniconda3/envs/fruitnet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-02 05:13:23,463 | INFO | PROJECT_ROOT=/home/diy-hus/fruitnet-chili-yield
2026-06-02 05:13:23,463 | INFO | DATA_ROOT=/mnt/f/fruitnet-chili-yield-data
2026-06-02 05:13:23,464 | INFO | Log file=logs/01_download_prepare_datasets_20260602_051323.log
DATA_ROOT: /mnt/f/fruitnet-chili-yield-data
DOWNLOAD_COCO: True
DOWNLOAD_MINNEAPPLE: False
MINNEAPPLE_URL exists: False
DOWNLOAD_GWHD: False
GWHD_ZENODO_RECORD: 5092309
2026-06-02 05:13:23,479 | INFO | Downloading http://images.cocodataset.org/zips/train2017.zip -> /mnt/f/fruitnet-chili-yield-data/raw/coco2017/train_images.zip


train_images.zip: 100%|##########| 18.0G/18.0G [57:49<00:00, 5.57MB/s]  


2026-06-02 06:11:13,586 | INFO | Downloading http://images.cocodataset.org/zips/val2017.zip -> /mnt/f/fruitnet-chili-yield-data/raw/coco2017/val_images.zip


val_images.zip: 100%|##########| 778M/778M [02:02<00:00, 6.68MB/s] 


...

annotations.zip: 100%|##########| 241M/241M [00:42<00:00, 5.95MB/s] 


2026-06-02 06:13:59,314 | INFO | Extracting /mnt/f/fruitnet-chili-yield-data/raw/coco2017/train_images.zip -> /mnt/f/fruitnet-chili-yield-data/raw/coco2017
2026-06-02 06:29:41,101 | INFO | Extracting /mnt/f/fruitnet-chili-yield-data/raw/coco2017/val_images.zip -> /mnt/f/fruitnet-chili-yield-data/raw/coco2017
2026-06-02 06:30:12,738 | INFO | Extracting /mnt/f/fruitnet-chili-yield-data/raw/coco2017/annotations.zip -> /mnt/f/fruitnet-chili-yield-data/raw/coco2017


Linking COCO val images and labels: 100%|##########| 5000/5000 [01:31<00:00, 54.78it/s] 


2026-06-02 07:11:43,178 | WARNING | minneapple: No JSON found in /mnt/f/fruitnet-chili-yield-data/raw/minneapple. If the dataset is already in YOLO format, place it in /mnt/f/fruitnet-chili-yield-data/processed/minneapple_yolo
2026-06-02 07:11:43,201 | WARNING | GWHD training CSV not found in /mnt/f/fruitnet-chili-yield-data/raw/gwhd
2026-06-02 07:11:43,212 | INFO | Wrote YAML file: configs/data/coco.yaml


{
  "coco_yaml": "/home/diy-hus/fruitnet-chili-yield/configs/data/coco.yaml",
  "minneapple_yaml": "/home/diy-hus/fruitnet-chili-yield/configs/data/minneapple.yaml",
  "gwhd_yaml": "/home/diy-hus/fruitnet-chili-yield/configs/data/gwhd.yaml",
  "coco_train_images": 118287,
  "coco_val_images": 5000,
  "minneapple_train_images": 0,
  "minneapple_val_images": 0,
  "gwhd_train_images": 0,
  "gwhd_val_images": 0,
  "log_file": "logs/01_download_prepare_datasets_20260602_051323.log"
}
Log file: logs/01_download_prepare_datasets_20260602_051323.log
